# Soup Kernel Challenge — Мерей: Kaggle T4 working copy

This notebook runs the working copy in `promaladoi/Decentrathon-6.0/Мерей`.
It checks out the pinned code commit `3c389e9f1259031123026c99057f78b3ad57b671`, then runs from
The full benchmark uses `--repeats 2`: base, solution, solution, base.
`/kaggle/working/repo/Мерей`. Set **Accelerator: GPU T4 x2** and **Internet: on**, then **Run All**.

The original dependency pins and the T4 check are preserved. The incompatible optional `torchao`
package is removed before tests and measurements. `harness/`, `tests/`, and `task/` remain unchanged.
Keep both GPUs idle during measurements. This is a development run; it does not establish final admission
until the complete rule checks pass. Current development results: [Kaggle development](https://www.kaggle.com/code/promaladoi/soup-kaggle-development).

The optional Kaggle secrets `GITHUB_TOKEN` (read-only) and `HF_TOKEN` work as in the original template.
The platform-repository submission template is preserved separately as `templates/kaggle_notebook.ipynb`.
Final submission still requires the repository and public, saved Kaggle version required by the challenge.


In [ ]:
REPO_URL = "https://github.com/promaladoi/Decentrathon-6.0.git"
COMMIT = "3c389e9f1259031123026c99057f78b3ad57b671"  # verified working-code snapshot

In [ ]:
import os, subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print(gpus)
assert "T4" in gpus, "Switch the accelerator to GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
# Remove an incompatible optional package before all tests and measurements.
!python -m pip uninstall -y torchao
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest

In [ ]:
import base64, os, shutil, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("No GITHUB_TOKEN secret, cloning without it:", type(exc).__name__)
auth = []
if token:  # the token goes in a header for this one command: never in a URL, a file or the output
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)  # always a fresh copy of your code
clone = subprocess.run(["git", *auth, "clone", "-q", REPO_URL, "/kaggle/working/repo"],
                       capture_output=True, text=True)
if clone.returncode:
    raise SystemExit("git clone failed: " + clone.stderr)
if COMMIT:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
%cd /kaggle/working/repo/Мерей
!python task/prepare_data.py
!python -m pytest solution -q

In [ ]:
!python harness/run_task.py --repeats 2

In [ ]:
!python harness/check_rules.py
!python harness/score.py runs/result.json

While iterating, a quicker run of your solution alone (checks 1 and 5-8 need the base, so they fail
in this mode): `python harness/run_task.py --only solution --out runs_try`. Add `--profile` to get a
table of every CUDA kernel's GPU time in `runs/solution_profile/profile.json`. Logs of every run
are in `runs/<run>/train.log`.